##### Data Generation

In [1]:
# .\payroll_venv\Scripts\activate inside cd 2_payroll-agentic-ai & deactivate command to come out
# python -m ipykernel install --user --name=payroll_venv --display-name "payroll_venv" kernel registered

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os

In [2]:
# employees synthetic data
fake = Faker()
np.random.seed(42)
random.seed(42)

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/documents", exist_ok=True)

NUM_EMPLOYEES = 250
PAY_PERIODS = pd.date_range(start="2025-07-01", periods=12, freq="SME")

departments = ["Operations", "Finance", "HR", "Sales", "IT", "Warehouse", "Customer Support"]
locations = ["CO", "TX", "CA", "NY", "FL"]
employment_types = ["Full-Time", "Part-Time", "Contractor"]

employees = []

for i in range(1, NUM_EMPLOYEES + 1):
    dept = random.choice(departments)
    emp_type = random.choices(employment_types, weights=[0.75, 0.15, 0.10])[0]
    
    base_salary = {
        "Operations": np.random.randint(55000, 90000),
        "Finance": np.random.randint(65000, 110000),
        "HR": np.random.randint(55000, 95000),
        "Sales": np.random.randint(50000, 120000),
        "IT": np.random.randint(75000, 140000),
        "Warehouse": np.random.randint(38000, 65000),
        "Customer Support": np.random.randint(40000, 70000),
    }[dept]
    
    employees.append({
        "employee_id": f"E{i:04d}",
        "employee_name": fake.name(),
        "department": dept,
        "job_title": fake.job(),
        "location_state": random.choice(locations),
        "employment_type": emp_type,
        "manager_id": f"M{random.randint(1, 25):03d}",
        "annual_salary": base_salary if emp_type != "Contractor" else 0,
        "hourly_rate": round(base_salary / 2080, 2) if emp_type != "Contractor" else np.random.randint(45, 120),
        "benefits_plan": random.choice(["Basic", "Standard", "Premium", "None"]),
        "hire_date": fake.date_between(start_date="-5y", end_date="-30d")
    })

employees_df = pd.DataFrame(employees)
employees_df.head()

,employee_id,employee_name,department,job_title,location_state,employment_type,manager_id,annual_salary,hourly_rate,benefits_plan,hire_date
0,E0001,Michelle Elliott,Warehouse,Statistician,CA,Full-Time,M008,54850,26.37,Standard,2021-08-31
1,E0002,Donna Martin,Finance,Research scientist (maths),FL,Full-Time,M003,109131,52.47,None,2025-08-12
2,E0003,Clifford Brown,Operations,Ambulance person,TX,Full-Time,M008,57433,27.61,Basic,2026-02-11
3,E0004,Leslie Newton,IT,Financial planner,FL,Full-Time,M014,139987,67.30,Standard,2024-11-29
4,E0005,Benjamin Stephens,Sales,Health visitor,CO,Full-Time,M025,117435,56.46,Standard,2023-01-25


In [3]:
# payroll synthetic data
payroll_records = []

for _, emp in employees_df.iterrows():
    normal_hours = 80 if emp["employment_type"] == "Full-Time" else 50
    
    for pay_date in PAY_PERIODS:
        hours_worked = max(0, np.random.normal(normal_hours, 8))
        overtime_hours = max(0, hours_worked - 80)
        
        gross_pay = emp["hourly_rate"] * hours_worked
        overtime_pay = emp["hourly_rate"] * 1.5 * overtime_hours
        
        bonus = random.choices([0, 250, 500, 1000], weights=[0.85, 0.08, 0.05, 0.02])[0]
        
        benefits_deduction = {
            "Basic": 75,
            "Standard": 150,
            "Premium": 275,
            "None": 0
        }[emp["benefits_plan"]]
        
        tax_rate = {
            "CO": 0.22,
            "TX": 0.18,
            "CA": 0.27,
            "NY": 0.26,
            "FL": 0.18
        }[emp["location_state"]]
        
        taxes = gross_pay * tax_rate
        net_pay = gross_pay + overtime_pay + bonus - benefits_deduction - taxes
        
        anomaly_type = "None"
        anomaly_flag = False
        
        # Inject realistic issues
        if random.random() < 0.04:
            overtime_hours += random.randint(10, 25)
            overtime_pay = emp["hourly_rate"] * 1.5 * overtime_hours
            anomaly_type = "Unusual Overtime"
            anomaly_flag = True
        
        if random.random() < 0.03:
            benefits_deduction *= -1
            anomaly_type = "Negative Deduction"
            anomaly_flag = True
        
        if random.random() < 0.03:
            bonus *= 2
            anomaly_type = "Duplicate Bonus"
            anomaly_flag = True
        
        payroll_records.append({
            "payroll_id": f"P{len(payroll_records)+1:06d}",
            "employee_id": emp["employee_id"],
            "pay_period_end": pay_date.date(),
            "hours_worked": round(hours_worked, 2),
            "overtime_hours": round(overtime_hours, 2),
            "gross_pay": round(gross_pay, 2),
            "overtime_pay": round(overtime_pay, 2),
            "bonus": round(bonus, 2),
            "benefits_deduction": round(benefits_deduction, 2),
            "taxes": round(taxes, 2),
            "net_pay": round(net_pay, 2),
            "anomaly_flag": anomaly_flag,
            "anomaly_type": anomaly_type
        })

payroll_df = pd.DataFrame(payroll_records)
payroll_df.head()

,payroll_id,employee_id,pay_period_end,hours_worked,overtime_hours,gross_pay,overtime_pay,bonus,benefits_deduction,taxes,net_pay,anomaly_flag,anomaly_type
0,P000001,E0001,2025-07-15,68.92,0.00,1817.34,0.00,0,150,490.68,1176.66,False,None
1,P000002,E0001,2025-07-31,71.14,0.00,1876.00,0.00,0,150,506.52,1219.48,False,None
2,P000003,E0001,2025-08-15,87.35,7.35,2303.54,290.90,0,150,621.95,1822.49,False,None
3,P000004,E0001,2025-08-31,84.42,4.42,2226.16,174.84,0,150,601.06,1649.94,False,None
4,P000005,E0001,2025-09-15,80.81,0.81,2131.07,32.20,500,150,575.39,1937.88,False,None


In [4]:
# payroll approvals synthetic data

approvals = []

for _, row in payroll_df.iterrows():
    if row["overtime_hours"] > 5:
        status = random.choices(["Approved", "Pending", "Rejected"], weights=[0.65, 0.25, 0.10])[0]
        
        approvals.append({
            "approval_id": f"A{len(approvals)+1:06d}",
            "employee_id": row["employee_id"],
            "payroll_id": row["payroll_id"],
            "approval_type": "Overtime",
            "approval_status": status,
            "submitted_date": row["pay_period_end"],
            "approved_by": f"M{random.randint(1, 25):03d}" if status == "Approved" else None
        })

approvals_df = pd.DataFrame(approvals)
approvals_df.head()


,approval_id,employee_id,payroll_id,approval_type,approval_status,submitted_date,approved_by
0,A000001,E0001,P000003,Overtime,Rejected,2025-08-15,NaN
1,A000002,E0001,P000008,Overtime,Approved,2025-10-31,M002
2,A000003,E0001,P000009,Overtime,Pending,2025-11-15,NaN
3,A000004,E0001,P000010,Overtime,Rejected,2025-11-30,NaN
4,A000005,E0002,P000016,Overtime,Pending,2025-08-31,NaN


In [5]:
# HR tickets synthetic data

ticket_reasons = [
    "Missing overtime pay",
    "Paycheck lower than expected",
    "Benefits deduction question",
    "Tax withholding question",
    "Missing bonus",
    "PTO deduction issue",
    "Incorrect hours worked",
    "Direct deposit issue"
]

tickets = []

sample_payroll = payroll_df.sample(180, random_state=42)

for _, row in sample_payroll.iterrows():
    tickets.append({
        "ticket_id": f"T{len(tickets)+1:06d}",
        "employee_id": row["employee_id"],
        "payroll_id": row["payroll_id"],
        "ticket_reason": random.choice(ticket_reasons),
        "ticket_description": fake.sentence(nb_words=14),
        "ticket_status": random.choices(["Open", "In Review", "Resolved"], weights=[0.25, 0.25, 0.50])[0],
        "created_date": row["pay_period_end"]
    })

tickets_df = pd.DataFrame(tickets)
tickets_df.head()

,ticket_id,employee_id,payroll_id,ticket_reason,ticket_description,ticket_status,created_date
0,T000001,E0151,P001802,Incorrect hours worked,Full get build side military growth floor incr...,Resolved,2025-07-31
1,T000002,E0100,P001191,PTO deduction issue,Fire not act central partner court especially ...,Open,2025-08-15
2,T000003,E0152,P001818,Tax withholding question,Although box center focus art theory entire fo...,Open,2025-09-30
3,T000004,E0021,P000252,Incorrect hours worked,Perform policy indicate responsibility check c...,Open,2025-12-31
4,T000005,E0209,P002506,Tax withholding question,No audience offer group recognize evening up t...,Resolved,2025-11-30


In [6]:
# save all CSV files
employees_df.to_csv("../data/raw/employees.csv", index=False)
payroll_df.to_csv("../data/raw/payroll_records.csv", index=False)
approvals_df.to_csv("../data/raw/manager_approvals.csv", index=False)
tickets_df.to_csv("../data/raw/payroll_tickets.csv", index=False)

print("Files created successfully:")
print("../data/raw/employees.csv")
print("../data/raw/payroll_records.csv")
print("../data/raw/manager_approvals.csv")
print("../data/raw/payroll_tickets.csv")

Files created successfully:
../data/raw/employees.csv
../data/raw/payroll_records.csv
../data/raw/manager_approvals.csv
../data/raw/payroll_tickets.csv


In [7]:
# Quick validation
print("Employees:", employees_df.shape)
print("Payroll records:", payroll_df.shape)
print("Approvals:", approvals_df.shape)
print("Tickets:", tickets_df.shape)

print("\nAnomaly counts:")
print(payroll_df["anomaly_type"].value_counts())

Employees: (250, 11)
Payroll records: (3000, 13)
Approvals: (686, 7)
Tickets: (180, 7)

Anomaly counts:
anomaly_type
None                  2710
Unusual Overtime       116
Duplicate Bonus         92
Negative Deduction      82
Name: count, dtype: int64


In [8]:
# policy doc synthetic data
payroll_policy = """
Company Payroll Policy

Employees are paid on a semi-monthly basis. Payroll is processed after timesheets, overtime records, bonuses, and deductions are validated.

Overtime Policy:
Full-time and part-time employees are eligible for overtime when they work more than 40 hours in a week. Overtime must be approved by the employee's manager before payroll processing. Unapproved overtime should be flagged for review before payment.

Benefits Deduction Policy:
Employees enrolled in Basic, Standard, or Premium benefits plans will have deductions applied each pay period. Employees with no benefits plan should not have benefits deductions.

Bonus Policy:
Bonuses must be approved by HR and Finance before payroll processing. Duplicate bonus entries should be flagged as high-risk payroll exceptions.

Payroll Exception Policy:
Payroll records should be reviewed when there are unusual overtime hours, negative deductions, duplicate bonuses, missing approvals, or major net pay changes compared with prior pay periods.

Employee Query Policy:
The payroll assistant may explain paycheck differences using payroll records and company policy. The assistant should not make final payroll changes. Any uncertain or disputed case must be escalated to HR or payroll specialists.
"""

with open("../data/documents/payroll_policy.txt", "w") as f:
    f.write(payroll_policy)

print("Policy document created.")

Policy document created.


##### Data Enrichment

In [9]:
# Add missing timesheet flags
payroll_df["timesheet_status"] = np.random.choice(
    ["Submitted", "Missing", "Late"],
    size=len(payroll_df),
    p=[0.85, 0.10, 0.05]
)

# Merge approvals into payroll
payroll_df = payroll_df.merge(
    approvals_df[["payroll_id", "approval_status"]],
    on="payroll_id",
    how="left"
)

payroll_df["approval_status"] = payroll_df["approval_status"].fillna("Not Required")

# add net pay change
payroll_df = payroll_df.sort_values(["employee_id", "pay_period_end"])
payroll_df["previous_net_pay"] = payroll_df.groupby("employee_id")["net_pay"].shift(1)
payroll_df["net_pay_change"] = payroll_df["net_pay"] - payroll_df["previous_net_pay"]
payroll_df.tail()

,payroll_id,employee_id,pay_period_end,hours_worked,overtime_hours,gross_pay,overtime_pay,bonus,benefits_deduction,taxes,net_pay,anomaly_flag,anomaly_type,timesheet_status,approval_status,previous_net_pay,net_pay_change
2995,P002996,E0250,2025-10-31,84.09,4.09,3773.12,275.28,0,75,679.16,3294.24,True,Duplicate Bonus,Submitted,Not Required,3505.93,-211.69
2996,P002997,E0250,2025-11-15,76.71,0.00,3441.76,0.00,0,75,619.52,2747.24,False,None,Submitted,Not Required,3294.24,-547.00
2997,P002998,E0250,2025-11-30,86.49,6.49,3880.85,436.87,0,75,698.55,3544.16,False,None,Submitted,Rejected,2747.24,796.92
2998,P002999,E0250,2025-12-15,81.11,1.11,3639.24,74.47,0,75,655.06,2983.64,False,None,Submitted,Not Required,3544.16,-560.52
2999,P003000,E0250,2025-12-31,75.97,0.00,3408.60,0.00,0,75,613.55,2720.06,False,None,Submitted,Not Required,2983.64,-263.58


In [10]:
# create risk score
def calculate_risk(row):
    score = 0
    if row["anomaly_flag"]:
        score += 3
    if row["timesheet_status"] == "Missing":
        score += 2
    if row["approval_status"] == "Pending":
        score += 2
    if abs(row["net_pay_change"] if pd.notnull(row["net_pay_change"]) else 0) > 500:
        score += 2 
    return score

payroll_df["risk_score"] = payroll_df.apply(calculate_risk, axis=1)
payroll_df["risk_level"] = pd.cut(
    payroll_df["risk_score"],
    bins=[-1, 1, 3, 10],
    labels=["Low", "Medium", "High"]
)

# issue summary explanation column
def generate_reason(row):
    reasons = []
    
    if row["anomaly_type"] != "None":
        reasons.append(row["anomaly_type"])
        
    if row["timesheet_status"] == "Missing":
        reasons.append("Missing timesheet")
        
    if row["approval_status"] == "Pending":
        reasons.append("Pending manager approval")
        
    if pd.notnull(row["net_pay_change"]) and abs(row["net_pay_change"]) > 500:
        reasons.append("Large change in net pay")
    
    return ", ".join(reasons) if reasons else "Normal"

payroll_df["issue_summary"] = payroll_df.apply(generate_reason, axis=1)
payroll_df.tail()

,payroll_id,employee_id,pay_period_end,hours_worked,overtime_hours,gross_pay,overtime_pay,bonus,benefits_deduction,taxes,net_pay,anomaly_flag,anomaly_type,timesheet_status,approval_status,previous_net_pay,net_pay_change,risk_score,risk_level,issue_summary
2995,P002996,E0250,2025-10-31,84.09,4.09,3773.12,275.28,0,75,679.16,3294.24,True,Duplicate Bonus,Submitted,Not Required,3505.93,-211.69,3,Medium,Duplicate Bonus
2996,P002997,E0250,2025-11-15,76.71,0.00,3441.76,0.00,0,75,619.52,2747.24,False,None,Submitted,Not Required,3294.24,-547.00,2,Medium,Large change in net pay
2997,P002998,E0250,2025-11-30,86.49,6.49,3880.85,436.87,0,75,698.55,3544.16,False,None,Submitted,Rejected,2747.24,796.92,2,Medium,Large change in net pay
2998,P002999,E0250,2025-12-15,81.11,1.11,3639.24,74.47,0,75,655.06,2983.64,False,None,Submitted,Not Required,3544.16,-560.52,2,Medium,Large change in net pay
2999,P003000,E0250,2025-12-31,75.97,0.00,3408.60,0.00,0,75,613.55,2720.06,False,None,Submitted,Not Required,2983.64,-263.58,0,Low,Normal


In [12]:
payroll_df.to_csv("../data/processed/payroll_enriched.csv", index=False)
print("Enriched payroll dataset created.")

Enriched payroll dataset created.


##### Data Dictionary

In [13]:
data_dictionary = """
========================
PAYROLL AI SYSTEM - DATA DICTIONARY
========================

------------------------
1. EMPLOYEES TABLE
------------------------

employee_id:
    Description: Unique identifier for each employee
    Type: String
    Example: E0001

employee_name:
    Description: Full name of the employee
    Type: String

department:
    Description: Business unit or department
    Type: Categorical
    Values: Operations, Finance, HR, Sales, IT, Warehouse, Customer Support

job_title:
    Description: Employee job role/title
    Type: String

location_state:
    Description: State where employee is located (used for tax calculation)
    Type: Categorical
    Values: CO, TX, CA, NY, FL

employment_type:
    Description: Type of employment
    Type: Categorical
    Values: Full-Time, Part-Time, Contractor

manager_id:
    Description: Manager identifier responsible for approvals
    Type: String

annual_salary:
    Description: Annual salary (only for salaried employees)
    Type: Float

hourly_rate:
    Description: Hourly rate used for payroll calculation
    Type: Float

benefits_plan:
    Description: Employee benefits enrollment plan
    Type: Categorical
    Values: Basic, Standard, Premium, None

hire_date:
    Description: Date employee joined the organization
    Type: Date

------------------------
2. PAYROLL RECORDS TABLE
------------------------

payroll_id:
    Description: Unique payroll transaction ID
    Type: String

employee_id:
    Description: Foreign key linking to employee
    Type: String

pay_period_end:
    Description: End date of payroll cycle
    Type: Date

hours_worked:
    Description: Total hours worked in the pay period
    Type: Float

overtime_hours:
    Description: Hours worked beyond standard threshold
    Type: Float

gross_pay:
    Description: Total earnings before deductions
    Type: Float

overtime_pay:
    Description: Additional pay for overtime hours
    Type: Float

bonus:
    Description: Bonus paid during the pay period
    Type: Float

benefits_deduction:
    Description: Amount deducted for benefits
    Type: Float

taxes:
    Description: Estimated tax deductions based on state
    Type: Float

net_pay:
    Description: Final take-home pay after deductions
    Type: Float

previous_net_pay:
    Description: Net pay from previous pay period
    Type: Float

net_pay_change:
    Description: Difference between current and previous net pay
    Type: Float

anomaly_flag:
    Description: Indicates whether payroll anomaly detected
    Type: Boolean

anomaly_type:
    Description: Type of anomaly detected
    Type: Categorical
    Values: None, Unusual Overtime, Negative Deduction, Duplicate Bonus

timesheet_status:
    Description: Status of employee timesheet submission
    Type: Categorical
    Values: Submitted, Missing, Late

approval_status:
    Description: Status of manager approval for payroll components
    Type: Categorical
    Values: Approved, Pending, Rejected, Not Required

risk_score:
    Description: Computed score indicating payroll risk level
    Type: Integer

risk_level:
    Description: Categorized risk level based on score
    Type: Categorical
    Values: Low, Medium, High

issue_summary:
    Description: AI-generated summary of payroll issues
    Type: String

------------------------
3. MANAGER APPROVALS TABLE
------------------------

approval_id:
    Description: Unique approval request identifier
    Type: String

employee_id:
    Description: Employee requesting approval
    Type: String

payroll_id:
    Description: Linked payroll record
    Type: String

approval_type:
    Description: Type of approval requested
    Type: Categorical
    Values: Overtime, PTO, Correction

approval_status:
    Description: Status of approval
    Type: Categorical
    Values: Approved, Pending, Rejected

submitted_date:
    Description: Date approval was requested
    Type: Date

approved_by:
    Description: Manager who approved the request
    Type: String

------------------------
4. PAYROLL TICKETS TABLE
------------------------

ticket_id:
    Description: Unique support ticket identifier
    Type: String

employee_id:
    Description: Employee raising the issue
    Type: String

payroll_id:
    Description: Related payroll record
    Type: String

ticket_reason:
    Description: Category of payroll issue
    Type: Categorical
    Values: Missing overtime pay, Pay discrepancy, Benefits issue, Tax issue, Missing bonus, PTO issue, Incorrect hours, Deposit issue

ticket_description:
    Description: Detailed description of issue
    Type: String

ticket_status:
    Description: Current status of ticket
    Type: Categorical
    Values: Open, In Review, Resolved

created_date:
    Description: Ticket creation date
    Type: Date

------------------------
5. DOCUMENTS (RAG DATA)
------------------------

payroll_policy.txt:
    Description: Company payroll policies including overtime, deductions, and exception handling

data_dictionary.txt:
    Description: Metadata describing all datasets and columns

------------------------
END OF DATA DICTIONARY
------------------------
"""

with open("../data/documents/data_dictionary.txt", "w") as f:
    f.write(data_dictionary)

print("Data dictionary created.")

Data dictionary created.


## Enhancement v2 — Rich HR/Payroll Policy Documents for RAG

This section creates multiple realistic policy/SOP documents instead of relying on one small policy file. These files will be uploaded to the Azure Blob `documents` container and indexed into Azure AI Search for stronger RAG retrieval.


In [14]:

# Enhancement v2: Generate richer HR/Payroll policy documents for RAG
import os
from pathlib import Path

DOCS_DIR = Path("../data/documents")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

policy_docs = {
    "pto_policy.txt": """
PTO POLICY

Purpose:
This policy explains employee eligibility, accrual, usage, approval, and carry-over rules for paid time off.

Eligibility:
Full-time employees become eligible for PTO after 90 days of continuous employment. Part-time employees may be eligible for prorated PTO based on scheduled hours. Contractors are not eligible for company PTO unless specified in their contract.

Accrual:
PTO is accrued each pay period. Accrual rates may vary by tenure, employment type, and location. Employees should review their pay statement or HR portal to confirm current PTO balance.

Carry-Over:
Employees may carry over up to 5 unused PTO days into the next calendar year. Any unused PTO beyond the carry-over limit may be forfeited unless protected by local law.

Approval:
PTO requests must be submitted in advance through the HR system. Manager approval is required before PTO is finalized. Extended leave requests may require additional HR review.

Payroll Impact:
Approved PTO is paid at the employee's regular base rate. Unapproved or unpaid leave may reduce gross pay for the applicable pay period.
""",

    "overtime_policy.txt": """
OVERTIME POLICY

Purpose:
This policy defines overtime eligibility, approval requirements, and payroll treatment.

Eligibility:
Non-exempt full-time and part-time employees are eligible for overtime when hours worked exceed applicable thresholds. Standard overtime is typically calculated at 1.5 times the regular hourly rate.

Approval Requirement:
Overtime should be approved by the employee's manager before payroll processing. Overtime without approval may be flagged for payroll review and may require manager validation.

Common Reasons for Overtime Review:
- Missing or late timesheets
- Manager approval pending
- Hours submitted after payroll cutoff
- Schedule mismatch
- Exception triggered by unusually high hours

Payroll Impact:
Approved overtime is included in the regular pay cycle. Pending or rejected overtime may delay or reduce overtime payment until reviewed.

Escalation:
Employees should contact their manager first for approval-related questions and payroll operations for payment calculation questions.
""",

    "payroll_deductions_policy.txt": """
PAYROLL DEDUCTIONS POLICY

Purpose:
This policy explains common payroll deductions and why net pay may differ from gross pay.

Deduction Categories:
Payroll deductions may include federal tax, state tax, Social Security, Medicare, health insurance, retirement contributions, garnishments, and benefit elections.

Benefits Deductions:
Benefit deductions may change when employees update coverage, add dependents, change plan options, or experience qualifying life events.

Tax Withholding:
Tax withholding can vary based on updated tax forms, bonus payments, overtime, supplemental wages, and state/local tax requirements.

Net Pay Changes:
Net pay may decrease even when gross pay is stable if deductions, withholding, or benefit contributions increase.

Employee Action:
Employees should review deduction details on the pay statement and contact payroll operations for unexplained changes.
""",

    "manager_approval_guidelines.txt": """
MANAGER APPROVAL GUIDELINES

Purpose:
This document explains manager responsibilities for approving payroll-related exceptions.

Approval Areas:
Managers may need to approve overtime, corrected hours, missed punches, PTO requests, shift adjustments, and payroll exceptions.

Approval Timing:
Approvals should be completed before payroll cutoff. Late approvals may cause payment delays or require off-cycle correction.

Review Checklist:
- Confirm submitted hours are accurate
- Validate overtime business need
- Confirm PTO or leave dates
- Review exception reason
- Approve or reject with clear notes

Operational Impact:
Pending manager approvals are a common cause of delayed overtime payments, PTO issues, and payroll tickets.
""",

    "payroll_escalation_process.txt": """
PAYROLL ESCALATION PROCESS

Purpose:
This SOP explains how payroll issues should be escalated and resolved.

Level 1: Employee Self-Service
Employees should first review pay statements, timesheets, PTO balances, and HR portal records.

Level 2: Manager Review
If the issue involves hours, overtime, schedule, or PTO approval, the employee should contact their manager for validation.

Level 3: Payroll Operations
Payroll operations investigates deduction issues, tax withholding, missing payments, direct deposit issues, and payroll calculation errors.

Level 4: HR Operations
HR operations reviews policy interpretation, leave eligibility, benefits eligibility, and employee classification questions.

SLA Guidance:
Standard payroll tickets should be reviewed within 2-3 business days. Urgent missing payment cases may require same-day review.
""",

    "benefits_policy.txt": """
BENEFITS POLICY

Purpose:
This policy explains employee benefits eligibility and payroll-related benefits deductions.

Eligibility:
Full-time employees are typically eligible for company benefits after the benefits waiting period. Part-time eligibility depends on scheduled hours and plan rules.

Benefit Elections:
Employees may elect medical, dental, vision, life insurance, disability coverage, and retirement contributions during onboarding or open enrollment.

Payroll Deductions:
Benefit elections are deducted from payroll based on selected plans and coverage level. Changes may appear in the next available pay cycle.

Qualifying Life Events:
Employees may update benefit elections after qualifying events such as marriage, birth, loss of coverage, or relocation.
""",

    "attendance_timesheet_policy.txt": """
ATTENDANCE AND TIMESHEET POLICY

Purpose:
This policy defines timesheet submission expectations and payroll impact.

Timesheet Submission:
Hourly employees must submit accurate timesheets before payroll cutoff. Missing or late timesheets may delay payroll processing.

Corrections:
Timesheet corrections require manager review. Corrections submitted after cutoff may be processed in a future pay cycle.

Payroll Impact:
Incorrect hours, missing punches, unapproved overtime, and late submissions may cause pay variance, delayed overtime, or payroll tickets.

Employee Responsibility:
Employees are responsible for reviewing hours before submission and promptly reporting discrepancies.
""",

    "data_dictionary.txt": """
PAYROLL AI SYSTEM DATA DICTIONARY

employees:
Contains employee profile data including employee_id, employee_name, department, location, employment_type, hire_date, and hourly_rate.

payroll_enriched:
Contains payroll transaction and risk-enriched records including payroll_id, employee_id, pay_period_end, hours_worked, overtime_hours, gross_pay, deductions, net_pay, anomaly_type, risk_score, and risk_level.

payroll_tickets:
Contains employee payroll support tickets including ticket_id, employee_id, payroll_id, ticket_reason, ticket_status, priority, created_date, and resolution_notes.

manager_approvals:
Contains payroll approval workflow data including approval_id, employee_id, payroll_id, approval_type, approval_status, submitted_date, and manager_comments.

Common joins:
employees.employee_id joins to payroll_enriched.employee_id, payroll_tickets.employee_id, and manager_approvals.employee_id.
payroll_enriched.payroll_id joins to payroll_tickets.payroll_id and manager_approvals.payroll_id.
"""
}

for filename, content in policy_docs.items():
    path = DOCS_DIR / filename
    path.write_text(content.strip(), encoding="utf-8")

print(f"Created {len(policy_docs)} policy/RAG documents in {DOCS_DIR}")
print("Documents:")
for filename in policy_docs:
    print("-", filename)


Created 8 policy/RAG documents in ..\data\documents
Documents:
- pto_policy.txt
- overtime_policy.txt
- payroll_deductions_policy.txt
- manager_approval_guidelines.txt
- payroll_escalation_process.txt
- benefits_policy.txt
- attendance_timesheet_policy.txt
- data_dictionary.txt
